# Percolation-Threshold Extractive Summarizer vs Fixed-Ratio Baselines

This notebook demonstrates the **percolation-threshold extractive summarizer** evaluated against fixed-ratio TF-score baselines on CNN/DailyMail documents.

**Method:** Build a vocabulary co-occurrence graph from the document. Greedily add TF-scored sentences until the giant connected component (GCC) of accumulated words reaches a critical fraction θ of the full document's GCC. Compare against fixed-ratio baselines that simply take the top-k% TF-scored sentences.

**Key finding:** Fixed-ratio baselines dominate on ROUGE-1 F1 (fixed_0.10: 0.286 vs percolation θ=0.6: 0.201), but percolation adapts dynamically to document structure — compression ratios vary substantially (θ=0.8: mean=0.629±0.092, range ~0.2–1.0).

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# rouge-score and loguru are NOT pre-installed on Colab — always install
_pip('rouge-score==0.1.2')
_pip('loguru==0.7.3')

# Core packages pre-installed on Colab; install locally to match Colab versions
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0',
         'networkx==3.6.1', 'nltk==3.9.1', 'datasets==4.0.0')

In [ ]:
import json
import math
import os
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import networkx as nx
from scipy import stats

# NLTK setup
import nltk
nltk_data_dir = "/tmp/nltk_data"
os.makedirs(nltk_data_dir, exist_ok=True)
nltk.data.path.insert(0, nltk_data_dir)
for pkg in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(pkg, download_dir=nltk_data_dir, quiet=True)

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6a9498-vocabulary-percolation-threshold-for-sel/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {len(data['examples'])} examples")

## Configuration

Tunable parameters for the demo run. Set to small values for quick execution; scale up for full evaluation.

In [ ]:
# --- Tunable parameters ---
N_DOCS = 5           # number of documents to process (original: 2000)
THETAS = [0.6, 0.7, 0.8, 0.9]   # percolation thresholds
FIXED_RATIOS = [0.10, 0.20, 0.30]  # fixed-ratio baselines

## Preprocessing

Tokenize each document into sentences, then extract content words (alpha, length ≥ 2, not stopwords) per sentence.

In [ ]:
from nltk.corpus import stopwords

def preprocess(text: str) -> tuple:
    """Return (raw_sentences, content_word_sets)."""
    stop = set(stopwords.words("english"))
    raw_sents = nltk.sent_tokenize(text)
    word_sets = []
    for s in raw_sents:
        words = set(
            w.lower() for w in nltk.word_tokenize(s)
            if w.isalpha() and len(w) >= 2 and w.lower() not in stop
        )
        word_sets.append(words)
    return raw_sents, word_sets

## Vocabulary Co-occurrence Graph

Build an undirected graph where nodes are content words and edges connect words that co-occur in the same sentence. Edge weight = number of co-occurrences.

In [ ]:
def build_vocab_graph(sentences_words: list) -> nx.Graph:
    G = nx.Graph()
    edge_counter: Counter = Counter()
    for words in sentences_words:
        wlist = sorted(words)
        for i in range(len(wlist)):
            for j in range(i + 1, len(wlist)):
                edge_counter[(wlist[i], wlist[j])] += 1
    G.add_nodes_from({w for ws in sentences_words for w in ws})
    for (u, v), wt in edge_counter.items():
        G.add_edge(u, v, weight=wt)
    return G


def gcc_size(G) -> int:
    if len(G) == 0:
        return 0
    return max((len(c) for c in nx.connected_components(G)), default=0)

## TF Scoring

Score each sentence by summing term frequencies of its content words. This is the signal used for greedy sentence selection in both the percolation and fixed-ratio summarizers.

In [ ]:
def compute_tf(sentences_words: list) -> dict:
    tf: Counter = Counter()
    for words in sentences_words:
        tf.update(words)
    return dict(tf)


def score_sentences(sentences_words: list, tf: dict) -> list:
    return [sum(tf.get(w, 0) for w in ws) for ws in sentences_words]

## Summarizers

**Percolation summarizer:** Greedily add highest-TF sentences until the GCC of accumulated words ≥ θ × full document GCC.

**Fixed-ratio summarizer:** Simply take the top `ceil(ratio × n)` TF-scored sentences.

In [ ]:
def percolation_summary(
    sentences_words: list,
    tf: dict,
    full_G,
    full_gcc: int,
    theta: float,
) -> tuple:
    n = len(sentences_words)
    if n == 0:
        return [], 0
    scores = score_sentences(sentences_words, tf)
    ranked = sorted(range(n), key=lambda i: -scores[i])
    if full_gcc == 0:
        return list(range(n)), n
    accumulated_words: set = set()
    selected = []
    k_star = n
    for idx in ranked:
        selected.append(idx)
        accumulated_words |= sentences_words[idx]
        sub = full_G.subgraph(accumulated_words)
        cur_gcc = max((len(c) for c in nx.connected_components(sub)), default=0)
        if cur_gcc / full_gcc >= theta:
            k_star = len(selected)
            break
    return sorted(selected), k_star


def fixed_ratio_summary(
    sentences_words: list, tf: dict, ratio: float
) -> tuple:
    n = len(sentences_words)
    k = max(1, math.ceil(ratio * n))
    scores = score_sentences(sentences_words, tf)
    ranked = sorted(range(n), key=lambda i: -scores[i])
    selected = sorted(ranked[:k])
    return selected, k

## ROUGE Evaluation

Evaluate ROUGE-1/2/L F1 for the selected sentences against the reference highlights.

In [ ]:
def evaluate(selected_indices: list, all_sentences: list, reference: str) -> dict:
    from rouge_score import rouge_scorer as rs_module
    scorer = rs_module.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    if not selected_indices:
        return {
            "rouge1_f": 0.0, "rouge1_r": 0.0, "rouge1_p": 0.0,
            "rouge2_f": 0.0, "rouge2_r": 0.0, "rouge2_p": 0.0,
            "rougeL_f": 0.0, "rougeL_r": 0.0, "rougeL_p": 0.0,
        }
    summary_text = " ".join(all_sentences[i] for i in sorted(selected_indices))
    scores = scorer.score(reference, summary_text)
    out = {}
    for k, v in scores.items():
        out[f"{k}_f"] = v.fmeasure
        out[f"{k}_r"] = v.recall
        out[f"{k}_p"] = v.precision
    return out

## Network Properties

Compute graph statistics (avg degree, clustering coefficient) to analyze how document topology correlates with percolation stopping behavior.

In [ ]:
import random

def network_properties(G) -> dict:
    if len(G) == 0:
        return {"avg_degree": 0.0, "clustering": 0.0, "n_nodes": 0, "n_edges": 0}
    degrees = [d for _, d in G.degree()]
    avg_degree = sum(degrees) / len(degrees)
    sample_nodes = random.sample(list(G.nodes()), min(200, len(G)))
    clustering = nx.average_clustering(G, nodes=sample_nodes)
    return {
        "avg_degree": round(avg_degree, 4),
        "clustering": round(clustering, 4),
        "n_nodes": len(G),
        "n_edges": G.number_of_edges(),
    }

## Per-Document Processing

Process each document: preprocess → build vocab graph → TF scoring → run all percolation thetas and fixed-ratio baselines → evaluate ROUGE.

In [ ]:
def process_doc(doc_id, article, highlights):
    raw_sents, word_sets = preprocess(article)
    # Filter empty sentences
    paired = [(s, ws) for s, ws in zip(raw_sents, word_sets) if ws]
    if not paired:
        return None
    raw_sents_filt = [p[0] for p in paired]
    word_sets_filt = [p[1] for p in paired]
    n = len(word_sets_filt)

    tf = compute_tf(word_sets_filt)
    full_G = build_vocab_graph(word_sets_filt)
    full_gcc = gcc_size(full_G)
    props = network_properties(full_G)

    result = {
        "doc_id": doc_id,
        "n_sentences": n,
        "full_gcc": full_gcc,
        "network": props,
        "percolation": {},
        "fixed": {},
    }

    for theta in THETAS:
        indices, k_star = percolation_summary(word_sets_filt, tf, full_G, full_gcc, theta)
        rouge = evaluate(indices, raw_sents_filt, highlights)
        compression = k_star / n if n > 0 else 1.0
        result["percolation"][str(theta)] = {
            "k_star": k_star,
            "compression_ratio": round(compression, 4),
            **rouge,
        }

    for ratio in FIXED_RATIOS:
        indices, k_used = fixed_ratio_summary(word_sets_filt, tf, ratio)
        rouge = evaluate(indices, raw_sents_filt, highlights)
        result["fixed"][str(ratio)] = {"k_used": k_used, **rouge}

    return result

In [ ]:
examples = data["examples"][:N_DOCS]
per_doc_results = []

for ex in examples:
    res = process_doc(ex["doc_id"], ex["article"], ex["highlights"])
    if res is not None:
        per_doc_results.append(res)
    print(f"Doc {ex['doc_id']}: n_sents={res['n_sentences']}, gcc={res['full_gcc']}, "
          f"perc_0.6 compression={res['percolation']['0.6']['compression_ratio']:.3f}, "
          f"rouge1_f={res['percolation']['0.6']['rouge1_f']:.3f} vs "
          f"fixed_0.1 rouge1_f={res['fixed']['0.1']['rouge1_f']:.3f}")

print(f"\nProcessed {len(per_doc_results)} documents")

## Aggregation

Compute mean/std ROUGE scores and compression statistics across all processed documents.

In [ ]:
def aggregate(per_doc: list) -> dict:
    agg = {"n_docs": len(per_doc)}

    for theta in THETAS:
        bucket = {}
        for metric in ["rouge1_f", "rouge2_f", "rougeL_f", "rouge1_r", "rouge2_r", "rougeL_r",
                        "compression_ratio"]:
            v = np.array([d["percolation"][str(theta)][metric] for d in per_doc
                           if str(theta) in d.get("percolation", {})])
            bucket[f"mean_{metric}"] = round(float(np.nanmean(v)), 4)
            bucket[f"std_{metric}"] = round(float(np.nanstd(v)), 4)
            if metric == "compression_ratio":
                bucket["min_compression"] = round(float(np.nanmin(v)), 4)
                bucket["max_compression"] = round(float(np.nanmax(v)), 4)
                bucket["p25_compression"] = round(float(np.nanpercentile(v, 25)), 4)
                bucket["p75_compression"] = round(float(np.nanpercentile(v, 75)), 4)
        agg[f"percolation_{theta}"] = bucket

    for ratio in FIXED_RATIOS:
        bucket = {}
        for metric in ["rouge1_f", "rouge2_f", "rougeL_f", "rouge1_r", "rouge2_r", "rougeL_r"]:
            v = np.array([d["fixed"][str(ratio)][metric] for d in per_doc
                           if str(ratio) in d.get("fixed", {})])
            bucket[f"mean_{metric}"] = round(float(np.nanmean(v)), 4)
            bucket[f"std_{metric}"] = round(float(np.nanstd(v)), 4)
        agg[f"fixed_{ratio}"] = bucket

    # Best percolation vs best fixed on rouge1_f
    best_theta = max(THETAS, key=lambda t: agg.get(f"percolation_{t}", {}).get("mean_rouge1_f", 0))
    best_ratio = max(FIXED_RATIOS, key=lambda r: agg.get(f"fixed_{r}", {}).get("mean_rouge1_f", 0))
    perc_r1 = np.array([d["percolation"][str(best_theta)]["rouge1_f"] for d in per_doc
                         if str(best_theta) in d.get("percolation", {})])
    fixed_r1 = np.array([d["fixed"][str(best_ratio)]["rouge1_f"] for d in per_doc
                          if str(best_ratio) in d.get("fixed", {})])
    n_paired = min(len(perc_r1), len(fixed_r1))
    if n_paired > 1:
        t_stat, p_val = stats.ttest_rel(perc_r1[:n_paired], fixed_r1[:n_paired])
        agg["statistical_tests"] = {
            "best_perc_theta": best_theta,
            "best_fixed_ratio": best_ratio,
            "best_perc_vs_best_fixed_rouge1_f": {
                "t_stat": round(float(t_stat), 4),
                "p_value": round(float(p_val), 6),
                "mean_diff": round(float(np.mean(perc_r1[:n_paired] - fixed_r1[:n_paired])), 4),
            }
        }

    # Correlations: percolation compression_ratio vs network props
    comp_08 = np.array([d["percolation"]["0.8"]["compression_ratio"] for d in per_doc
                         if "0.8" in d.get("percolation", {})])
    avg_deg = np.array([d["network"]["avg_degree"] for d in per_doc
                         if "0.8" in d.get("percolation", {})])
    clust = np.array([d["network"]["clustering"] for d in per_doc
                       if "0.8" in d.get("percolation", {})])

    def corr_dict(x, y):
        if len(x) < 3:
            return {}
        pr, pp = stats.pearsonr(x, y)
        sr, sp = stats.spearmanr(x, y)
        return {"pearson_r": round(float(pr), 4), "pearson_p": round(float(pp), 6),
                "spearman_r": round(float(sr), 4), "spearman_p": round(float(sp), 6)}

    agg["correlation_perc_ratio_vs_avg_degree"] = corr_dict(comp_08, avg_deg)
    agg["correlation_perc_ratio_vs_clustering"] = corr_dict(comp_08, clust)

    return agg


agg = aggregate(per_doc_results)
print(f"Aggregated {agg['n_docs']} documents")

## Results

Summary table and visualizations comparing percolation vs fixed-ratio methods.

In [ ]:
# Print results table
print(f"{'Method':<22} {'ROUGE-1 F1':>12} {'ROUGE-2 F1':>12} {'ROUGE-L F1':>12} {'Compression':>12}")
print("-" * 72)
for theta in THETAS:
    k = f"percolation_{theta}"
    b = agg.get(k, {})
    comp = f"{b.get('mean_compression_ratio',0):.3f}±{b.get('std_compression_ratio',0):.3f}"
    print(f"Percolation θ={theta:<5} {b.get('mean_rouge1_f',0):>12.4f} {b.get('mean_rouge2_f',0):>12.4f} {b.get('mean_rougeL_f',0):>12.4f} {comp:>12}")

print("-" * 72)
for ratio in FIXED_RATIOS:
    k = f"fixed_{ratio}"
    b = agg.get(k, {})
    print(f"Fixed ratio={ratio:<8} {b.get('mean_rouge1_f',0):>12.4f} {b.get('mean_rouge2_f',0):>12.4f} {b.get('mean_rougeL_f',0):>12.4f} {'(fixed)':>12}")

if "statistical_tests" in agg:
    st = agg["statistical_tests"]["best_perc_vs_best_fixed_rouge1_f"]
    print(f"\nBest percolation (θ={agg['statistical_tests']['best_perc_theta']}) vs "
          f"best fixed (ratio={agg['statistical_tests']['best_fixed_ratio']}):")
    print(f"  Mean diff = {st['mean_diff']:.4f}, t = {st['t_stat']:.4f}, p = {st['p_value']:.6f}")

# --- Visualizations ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: ROUGE-1 F1 comparison
ax = axes[0]
labels_perc = [f"θ={t}" for t in THETAS]
vals_perc = [agg.get(f"percolation_{t}", {}).get("mean_rouge1_f", 0) for t in THETAS]
labels_fixed = [f"{r:.0%}" for r in FIXED_RATIOS]
vals_fixed = [agg.get(f"fixed_{r}", {}).get("mean_rouge1_f", 0) for r in FIXED_RATIOS]
x = range(len(labels_perc) + len(labels_fixed))
colors = ["#4C72B0"] * len(labels_perc) + ["#DD8452"] * len(labels_fixed)
ax.bar(x, vals_perc + vals_fixed, color=colors)
ax.set_xticks(list(x))
ax.set_xticklabels(labels_perc + labels_fixed, rotation=15)
ax.set_ylabel("Mean ROUGE-1 F1")
ax.set_title("ROUGE-1 F1: Percolation (blue) vs Fixed (orange)")
ax.grid(axis="y", alpha=0.3)

# Plot 2: Compression ratio by theta
ax = axes[1]
means_comp = [agg.get(f"percolation_{t}", {}).get("mean_compression_ratio", 0) for t in THETAS]
stds_comp = [agg.get(f"percolation_{t}", {}).get("std_compression_ratio", 0) for t in THETAS]
ax.bar(range(len(THETAS)), means_comp, yerr=stds_comp, capsize=5, color="#4C72B0", alpha=0.8)
ax.set_xticks(range(len(THETAS)))
ax.set_xticklabels([f"θ={t}" for t in THETAS])
ax.set_ylabel("Mean Compression Ratio")
ax.set_title("Percolation Compression Ratio (mean ± std)")
ax.grid(axis="y", alpha=0.3)

# Plot 3: Per-doc ROUGE-1 F1 scatter: percolation theta=0.6 vs fixed 0.10
ax = axes[2]
perc_vals = [d["percolation"]["0.6"]["rouge1_f"] for d in per_doc_results]
fixed_vals = [d["fixed"]["0.1"]["rouge1_f"] for d in per_doc_results]
ax.scatter(fixed_vals, perc_vals, alpha=0.7, color="#4C72B0")
lim = max(max(perc_vals + fixed_vals) + 0.05, 0.4)
ax.plot([0, lim], [0, lim], "k--", alpha=0.4, label="y=x")
ax.set_xlabel("Fixed 10% ROUGE-1 F1")
ax.set_ylabel("Percolation θ=0.6 ROUGE-1 F1")
ax.set_title("Per-document: Percolation vs Fixed")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("results_summary.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved results_summary.png")